In [3]:
import warnings

# Suppress all UserWarning messages
warnings.filterwarnings("ignore", category=UserWarning)

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [4]:
data = pd.read_csv('../data/Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
# In this estimated salary is going to be the output feature and remaining will be the independent feature
# That's the reason why output feature is a continuous variable and it then turns out to be a regression problem
data = data.drop(['RowNumber', 'CustomerId','Surname'],axis=1)


In [6]:
# Encode categorical variables
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [7]:
# Encode the Geography using One-Hot encoder
onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [8]:
# Combining onehot encoded columns with the data
data = pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [9]:
# Splitting the data into independent features and target feature
X = data.drop('EstimatedSalary', axis=1)
y = data['EstimatedSalary']



In [10]:
# Splitting the data in train-test set
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2,random_state=42)

In [11]:
# Scaling down the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [12]:
with open('label_encoder_gender_reg.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('onehot_encoder_geo_reg.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)

with open('scaler_reg.pkl','wb') as file:
    pickle.dump(scaler,file)

In [13]:
# Train ANN for Regression

import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense
from keras.callbacks import EarlyStopping, TensorBoard 
import datetime

In [14]:
# Building the model
model = Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)), # Input -> Hidden layer 1
    Dense(32,activation='relu'), # Hidden layer 1 -> Hidden layer 2
    Dense(1) # Hidden layer 2 -> Output Default activation function applied here is Linear Activation function
])

In [15]:
# Compiling the model
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# Setting up TensorBoard
log_dir = 'regressionlogs/fit/' +datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
tensorboard_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [17]:
# Setting up early stopping
early_stopping_callback = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)


In [18]:
history = model.fit(
    X_train, y_train,
    validation_data = (X_test,y_test),
    epochs=100,
    callbacks=[early_stopping_callback,tensorboard_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 100379.3125 - mae: 100379.3125 - val_loss: 98527.8750 - val_mae: 98527.8750
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 99669.5703 - mae: 99669.5703 - val_loss: 97084.8281 - val_mae: 97084.8281
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 97146.3594 - mae: 97146.3594 - val_loss: 93376.0859 - val_mae: 93376.0859
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 92149.1875 - mae: 92149.1875 - val_loss: 87115.7578 - val_mae: 87115.7578
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 84845.2891 - mae: 84845.2891 - val_loss: 78995.5312 - val_mae: 78995.5312
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 76145.2500 - mae: 76145.2500 - val_loss: 70421.1250 - val_mae: 70421.1250
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 67491.2578 - mae: 67491.2578 - val_loss: 62683.2734 - val_mae: 62683.2734
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step 

In [19]:
%load_ext tensorboard

In [20]:
%tensorboard --logdir regressionlogs/fit

In [21]:
# Evaluate model on test data
test_loss,test_mae = model.evaluate(X_test,y_test)
print(f"Test MAE : {test_mae:.2f}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 50142.5703 - mae: 50142.5703
Test MAE : 50142.57


In [22]:
model.save('regression_model.keras')